# FastSpeech2 — **Local synthesis / testing** (VSCode)

Load a trained checkpoint and synthesize Bulgarian speech **on your own machine** — no Colab, no GPU required (CPU is fine for inference).

**What you need (3 files):**
1. a **checkpoint** `…/<STEP>.pth.tar` (download one from `MyDrive/fs2_bg_real/output/ckpt/Bulgarian/`)
2. **`stats.json`** — the pitch/energy normalisation stats the model needs (inside `MyDrive/fs2_bg_real/preprocessed_Bulgarian.zip` → `Bulgarian/stats.json`)
3. the **HiFi-GAN** vocoder (ships zipped in the repo; section 3 unzips it)

> Inference does **not** need MFA, `pyworld`, the dataset, or `speakers.json` (the model is single-speaker). Just the model weights + `stats.json` + the vocoder.

Run the cells top to bottom.

## 1. Point the notebook at the repo root
Walks up from the current folder until it finds `config/Bulgarian/`, then `chdir`s there so all the relative paths below resolve no matter where the notebook lives.

In [ ]:
import os
d = os.getcwd()
while not os.path.exists(os.path.join(d, "config", "Bulgarian", "preprocess.yaml")):
    parent = os.path.dirname(d)
    if parent == d:
        raise RuntimeError("FastSpeech2 repo root not found - open this notebook from inside the repo.")
    d = parent
os.chdir(d)
print("repo root:", os.getcwd())

## 2. Install inference dependencies
A one-time install into the notebook's Python. Inference-only set (no MFA / pyworld / tgt / sklearn).

In [ ]:
%pip install -q torch numpy scipy pyyaml matplotlib tqdm soundfile librosa

## 3. Unzip the HiFi-GAN vocoder
Extracts `hifigan/generator_universal.pth.tar` (cross-platform; uses Python's `zipfile`, so it works on Windows without `unzip`).

In [ ]:
import zipfile, os
src = "hifigan/generator_universal.pth.tar.zip"
dst = "hifigan/generator_universal.pth.tar"
if os.path.exists(dst):
    print("already extracted:", dst)
else:
    with zipfile.ZipFile(src) as z:
        z.extractall("hifigan")
    print("extracted ->", dst, "| exists:", os.path.exists(dst))

## 4. Put your checkpoint + `stats.json` in place
Run this to create the folders, then copy the two files you downloaded from Drive into them:

- the checkpoint  → `output/ckpt/Bulgarian/<STEP>.pth.tar`  (keep the step number in the name)
- `stats.json`    → `preprocessed_data/Bulgarian/stats.json`

You can drag-and-drop them in the VSCode Explorer.

In [ ]:
import os
for folder in ["output/ckpt/Bulgarian", "preprocessed_data/Bulgarian", "local_samples"]:
    os.makedirs(folder, exist_ok=True)
print("folders ready:")
print("  checkpoint  -> output/ckpt/Bulgarian/<STEP>.pth.tar")
print("  stats.json  -> preprocessed_data/Bulgarian/stats.json")

## 5. Set the checkpoint step & verify the files
Set `CKPT_STEP` to the step number of the `.pth.tar` you copied in (e.g. a file named `25000.pth.tar` → `CKPT_STEP = 25000`).

In [ ]:
CKPT_STEP = 25000   # <-- step number of the checkpoint you placed in output/ckpt/Bulgarian/

import os
required = {
    "checkpoint": f"output/ckpt/Bulgarian/{CKPT_STEP}.pth.tar",
    "stats.json": "preprocessed_data/Bulgarian/stats.json",
    "vocoder":    "hifigan/generator_universal.pth.tar",
}
ok = True
for label, path in required.items():
    present = os.path.exists(path)
    ok = ok and present
    print(("OK      " if present else "MISSING "), f"{label:11s}", path)
assert ok, "Copy the missing file(s) into place, then re-run this cell."
print("\nAll set - continue to section 6.")

## 6. Load the model + vocoder *(run once)*
Loads the checkpoint into memory and defines a `synth(...)` helper. Reloading is only needed if you change `CKPT_STEP`.

In [ ]:
import argparse, yaml, torch, numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

preprocess_config = yaml.load(open("config/Bulgarian/preprocess.yaml"), Loader=yaml.FullLoader)
model_config      = yaml.load(open("config/Bulgarian/model.yaml"),      Loader=yaml.FullLoader)
train_config      = yaml.load(open("config/Bulgarian/train.yaml"),      Loader=yaml.FullLoader)
configs = (preprocess_config, model_config, train_config)

from utils.model import get_model, get_vocoder, vocoder_infer
from utils.tools import to_device
from synthesize import preprocess_bulgarian   # same text pipeline as training

args = argparse.Namespace(restore_step=CKPT_STEP)
model = get_model(args, configs, device, train=False)
vocoder = get_vocoder(model_config, device)
print("loaded model from step", CKPT_STEP, "| params:",
      sum(p.numel() for p in model.parameters()))

import os
from scipy.io import wavfile
from IPython.display import Audio, display
os.makedirs("local_samples", exist_ok=True)

def synth(text, pitch=1.0, energy=1.0, duration=1.0, name=None):
    """Synthesize one Bulgarian sentence; saves a wav and plays it inline.
    pitch/energy: >1 raises, <1 lowers. duration: >1 slower, <1 faster."""
    ids = raw_texts = [text[:100]]
    speakers  = np.array([0])
    texts     = np.array([preprocess_bulgarian(text, preprocess_config)])
    text_lens = np.array([len(texts[0])])
    batch = to_device((ids, raw_texts, speakers, texts, text_lens, max(text_lens)), device)
    with torch.no_grad():
        output = model(*(batch[2:]), p_control=pitch, e_control=energy, d_control=duration)
    mel = output[1].transpose(1, 2)
    lengths = output[9] * preprocess_config["preprocessing"]["stft"]["hop_length"]
    wav = vocoder_infer(mel, vocoder, model_config, preprocess_config, lengths=lengths)[0]
    sr = preprocess_config["preprocessing"]["audio"]["sampling_rate"]
    name = name or (text[:40].strip().replace(" ", "_") or "sample")
    out = os.path.join("local_samples", f"{name}.wav")
    wavfile.write(out, sr, wav)
    print("saved:", out)
    display(Audio(wav, rate=sr))
    return out

## 7. 🔊 Synthesize — edit the text and run
Re-run this cell as many times as you like with different text. The controls are optional:
`duration` > 1 speaks slower, `pitch`/`energy` > 1 raise pitch/volume.

In [ ]:
TEXT = "Здравей! Как си днес?"

synth(TEXT, pitch=1.0, energy=1.0, duration=1.0)

## 8. (Optional) Make an **inference-only** checkpoint

**What's inside a training checkpoint?** Each `<STEP>.pth.tar` saved during training holds three things:

| key | what it is | needed for inference? |
|-----|------------|-----------------------|
| `model` | the FastSpeech2 weights | ✅ yes — the only part you need |
| `optimizer` | Adam state: per-weight momentum buffers (`exp_avg`, `exp_avg_sq`) | ❌ no — training only |
| `lr_plateau` | ReduceLROnPlateau bookkeeping (best loss, bad-epoch count, scale) | ❌ no — training only |

The **learning rate is *not* stored as a number** — it's recomputed from the Noam schedule using the step (which comes from the filename / `--restore_step`), then scaled by `lr_plateau`. The bulk of the non-weight bytes is the Adam optimizer's two momentum tensors per parameter, so dropping it roughly **halves** the file.

The cell below writes a slim `{model: ...}` checkpoint. It still loads for inference (the loader only reads the `model` key when `train=False`) but **cannot resume training** — that's the trade-off.

In [ ]:
import torch, os

SRC = f"output/ckpt/Bulgarian/{CKPT_STEP}.pth.tar"
DST = f"output/ckpt/Bulgarian/{CKPT_STEP}_inference.pth.tar"

ck = torch.load(SRC, map_location="cpu", weights_only=False)
print("keys in full checkpoint:", list(ck.keys()))
torch.save({"model": ck["model"]}, DST)

print(f"full : {os.path.getsize(SRC)/1e6:7.1f} MB  {SRC}")
print(f"slim : {os.path.getsize(DST)/1e6:7.1f} MB  {DST}")
print("\nTo use the slim file for inference, rename it to <STEP>.pth.tar and set CKPT_STEP accordingly.")